# Overview — Default of Credit Card Clients Dataset

**Source:** UCI Machine Learning Repository — ID 350  
**Objective:** Understand the data before injecting artificial missingness.

This dataset contains information on **30,000 credit card clients** in Taiwan from April to September 2005.  
It covers payment history, bill amounts, and demographic information.

The target variable is **default** — whether the client defaulted on their payment next month:
- **1** = Yes (defaulted)  
- **0** = No (did not default)

**Business question:** Can we predict credit card default from client demographics, credit limit, and payment history?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ucimlrepo import fetch_ucirepo

## 1. Load the Dataset

In [ ]:
default_credit = fetch_ucirepo(id=350)
df = pd.concat([default_credit.data.features.copy(),
                default_credit.data.targets.rename(
                    columns={default_credit.data.targets.columns[0]: "default"})],
               axis=1)

print(f"Dimensions: {df.shape[0]} observations × {df.shape[1]} variables")
df.head(10)

## 2. Variable Types and Missing Values

In [ ]:
info = pd.DataFrame({
    "dtype":     df.dtypes,
    "n_unique":  df.nunique(),
    "missing":   df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "min":       df.min(),
    "max":       df.max(),
    "example":   df.iloc[0]
})
info

### Variable Codebook

| Variable | Type | Description |
|---|---|---|
| `LIMIT_BAL` | Continuous | Credit limit (NT dollar) |
| `SEX` | Binary | 1=Male, 2=Female |
| `EDUCATION` | Ordinal | 1=Grad school, 2=University, 3=High school, 4=Others |
| `MARRIAGE` | Categorical | 1=Married, 2=Single, 3=Others |
| `AGE` | Continuous | Age in years |
| `PAY_0`–`PAY_6` | Ordinal | Repayment status (April–September 2005): -1=pay duly, 1=delay 1 month, ..., 9=delay 9+ months |
| `BILL_AMT1`–`BILL_AMT6` | Continuous | Bill statement amount (NT dollar), Sept–April |
| `PAY_AMT1`–`PAY_AMT6` | Continuous | Amount paid (NT dollar), Sept–April |
| `default` | Target (0/1) | 1 = defaulted next month |

In [ ]:
demo_cols    = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
pay_cols     = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols    = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
                "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
                "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]
target_col   = "default"

print(f"Demographic:    {demo_cols}")
print(f"Payment status: {pay_cols}")
print(f"Bill amounts:   {bill_cols}")
print(f"Paid amounts:   {pay_amt_cols}")

## 3. Descriptive Statistics

In [ ]:
print("=== Demographic & Credit ===")
df[demo_cols].describe().T.round(2)

In [ ]:
print("=== Bill Amounts ===")
display(df[bill_cols].describe().T.round(0))
print("\n=== Payment Amounts ===")
df[pay_amt_cols].describe().T.round(0)

## 4. Target Variable — Default Payment

In [ ]:
vc = df[target_col].value_counts().sort_index()
labels = {0: "0 – No default", 1: "1 – Default"}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Count", "Percentage"],
                    specs=[[{"type": "bar"}, {"type": "pie"}]])

fig.add_trace(go.Bar(x=[labels[k] for k in vc.index], y=vc.values,
                     marker_color=["steelblue", "coral"],
                     text=vc.values, textposition="outside",
                     showlegend=False), row=1, col=1)

fig.add_trace(go.Pie(labels=[labels[k] for k in vc.index], values=vc.values,
                     marker_colors=["steelblue", "coral"],
                     hole=0.3), row=1, col=2)

fig.update_layout(title_text="Distribution of Default (target)", height=400)
fig.show()

default_rate = df[target_col].mean() * 100
print(f"Default rate: {default_rate:.1f}%  ({vc[1]:,} out of {len(df):,} clients)")

## 5. Demographic Variable Distributions

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Distribution of LIMIT_BAL", "Boxplot of LIMIT_BAL",
                                    "Distribution of AGE", "Boxplot of AGE"])

fig.add_trace(go.Histogram(x=df["LIMIT_BAL"], nbinsx=50,
                           marker_color="steelblue", showlegend=False), row=1, col=1)
fig.add_trace(go.Box(y=df["LIMIT_BAL"], marker_color="steelblue",
                     boxmean=True, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=df["AGE"], nbinsx=40,
                           marker_color="coral", showlegend=False), row=2, col=1)
fig.add_trace(go.Box(y=df["AGE"], marker_color="coral",
                     boxmean=True, showlegend=False), row=2, col=2)

fig.update_layout(title_text="Credit Limit and Age Distributions", height=600)
fig.show()

print(df[["LIMIT_BAL", "AGE"]].describe().T.round(2))

In [ ]:
sex_labels  = {1: "Male", 2: "Female"}
edu_labels  = {1: "Grad school", 2: "University", 3: "High school", 4: "Others", 5: "Unknown", 6: "Unknown2"}
mar_labels  = {0: "Others", 1: "Married", 2: "Single", 3: "Others2"}

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["SEX", "EDUCATION", "MARRIAGE"])

for col, lmap, c in [("SEX", sex_labels, 1), ("EDUCATION", edu_labels, 2), ("MARRIAGE", mar_labels, 3)]:
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(
        x=[lmap.get(k, str(k)) for k in vc.index],
        y=vc.values,
        marker_color="mediumpurple",
        text=vc.values, textposition="outside",
        showlegend=False
    ), row=1, col=c)

fig.update_layout(title_text="Categorical Variable Frequencies", height=400)
fig.show()

## 6. Payment Status History (PAY variables)

In [ ]:
month_labels = ["Sep", "Aug", "Jul", "Jun", "May", "Apr"]

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f"PAY ({m})" for m in month_labels])

for i, (col, month) in enumerate(zip(pay_cols, month_labels)):
    r = i // 3 + 1
    c = i % 3 + 1
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="steelblue", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Payment Status Distribution by Month",
                  height=500)
fig.show()

# Proportion paying on time (PAY_0 <= 0) across months
on_time = {col: (df[col] <= 0).mean() * 100 for col in pay_cols}
print("% clients paying on time (status ≤ 0) by month:")
for col, pct in zip(month_labels, on_time.values()):
    print(f"  {col}: {pct:.1f}%")

## 7. Bill and Payment Amount Trends Over Time

In [ ]:
bill_means = df[bill_cols].mean()
pay_means  = df[pay_amt_cols].mean()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Average Bill Amount by Month",
                                    "Average Payment Amount by Month"])

fig.add_trace(go.Scatter(x=month_labels, y=bill_means.values,
                         mode="lines+markers",
                         marker=dict(size=8, color="coral"),
                         line=dict(color="coral"),
                         name="Avg Bill"), row=1, col=1)

fig.add_trace(go.Scatter(x=month_labels, y=pay_means.values,
                         mode="lines+markers",
                         marker=dict(size=8, color="mediumseagreen"),
                         line=dict(color="mediumseagreen"),
                         name="Avg Payment"), row=1, col=2)

fig.update_yaxes(title_text="NT Dollar", row=1, col=1)
fig.update_yaxes(title_text="NT Dollar", row=1, col=2)
fig.update_layout(title_text="Bill and Payment Trends (Sept → Apr)",
                  height=400)
fig.show()

## 8. Feature–Target Relationships

### 8.1 Demographic Variables vs Default Rate

In [ ]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Default Rate by SEX",
                                    "Default Rate by EDUCATION",
                                    "Default Rate by MARRIAGE"])

for col, lmap, c in [("SEX", sex_labels, 1), ("EDUCATION", edu_labels, 2), ("MARRIAGE", mar_labels, 3)]:
    grp = df.groupby(col)[target_col].mean() * 100
    fig.add_trace(go.Bar(
        x=[lmap.get(k, str(k)) for k in grp.index],
        y=grp.values.round(1),
        text=[f"{v:.1f}%" for v in grp.values],
        textposition="outside",
        marker_color="coral",
        showlegend=False
    ), row=1, col=c)
    fig.update_yaxes(title_text="Default rate (%)", row=1, col=c)

fig.update_layout(title_text="Default Rate by Categorical Features", height=400)
fig.show()

In [ ]:
# LIMIT_BAL and AGE vs default
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["LIMIT_BAL by Default", "AGE by Default"])

for val, color in [(0, "steelblue"), (1, "coral")]:
    sub = df[df[target_col] == val]
    label = labels[val]
    fig.add_trace(go.Box(y=sub["LIMIT_BAL"], name=label,
                         marker_color=color, boxmean=True), row=1, col=1)
    fig.add_trace(go.Box(y=sub["AGE"], name=label,
                         marker_color=color, boxmean=True, showlegend=False), row=1, col=2)

fig.update_layout(title_text="Credit Limit and Age by Default Status", height=450)
fig.show()

print(df.groupby(target_col)[["LIMIT_BAL", "AGE"]].mean().round(2))

### 8.2 Payment Status vs Default Rate

In [ ]:
# Default rate by PAY_0 status (most recent month)
pay0_default = df.groupby("PAY_0")[target_col].mean() * 100

fig = px.bar(x=pay0_default.index, y=pay0_default.values,
             color=pay0_default.values,
             color_continuous_scale="Reds",
             title="Default Rate by PAY_0 (September repayment status)",
             labels={"x": "PAY_0 status", "y": "Default rate (%)"})
fig.update_layout(coloraxis_showscale=False)
fig.add_hline(y=df[target_col].mean()*100, line_dash="dash",
              annotation_text=f"Overall avg: {df[target_col].mean()*100:.1f}%")
fig.show()

print("Default rate by PAY_0:")
print(pay0_default.round(1).to_string())

## 9. Correlation Matrix

In [ ]:
corr = df.corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale="RdBu",
    zmid=0,
    colorbar=dict(title="r")
))
fig.update_layout(title="Pearson Correlation Matrix",
                  width=900, height=850)
fig.show()

# Strongest correlations with target
corr_target = corr[target_col].drop(target_col).sort_values()
print("Top 5 positive correlations with default:")
print(corr_target.tail(5).to_string())
print("\nTop 5 negative correlations with default:")
print(corr_target.head(5).to_string())

## 10. Feature Correlation with Default

In [ ]:
feature_corr = df.corr()[target_col].drop(target_col).sort_values()

fig = px.bar(x=feature_corr.values, y=feature_corr.index,
             orientation="h",
             color=feature_corr.values,
             color_continuous_scale="RdBu",
             color_continuous_midpoint=0,
             title="Pearson Correlation with Default",
             labels={"x": "Correlation", "y": "Variable"})
fig.update_layout(coloraxis_showscale=False, height=700)
fig.show()

## 11. Credit Limit vs Age — Colored by Default

In [ ]:
df_sample = df.sample(n=3000, random_state=42)

fig = px.scatter(df_sample, x="AGE", y="LIMIT_BAL",
                 color=df_sample[target_col].astype(str),
                 color_discrete_map={"0": "steelblue", "1": "coral"},
                 opacity=0.5,
                 title="Age vs Credit Limit — colored by Default (sample n=3000)",
                 labels={"AGE": "Age", "LIMIT_BAL": "Credit Limit (NT$)",
                         "color": "Default"})
fig.show()

## 12. Bill vs Payment Amounts — Sep 2005

In [ ]:
df_sample = df.sample(n=3000, random_state=42)

fig = px.scatter(df_sample, x="BILL_AMT1", y="PAY_AMT1",
                 color=df_sample[target_col].astype(str),
                 color_discrete_map={"0": "steelblue", "1": "coral"},
                 opacity=0.5,
                 title="Bill Amount vs Payment Amount (Sep 2005) — colored by Default",
                 labels={"BILL_AMT1": "Bill Amount Sep (NT$)",
                         "PAY_AMT1": "Payment Amount Sep (NT$)",
                         "color": "Default"})
fig.show()

## 13. Quick Conclusions

| Finding | Detail |
|---|---|
| **Default rate** | 22.1% — moderate class imbalance, but manageable |
| **Strongest predictor** | `PAY_0` (September repayment status): delayed payments are the clearest signal of upcoming default |
| **Credit limit** | Defaulters have significantly lower credit limits — lower limits may be assigned to riskier clients |
| **Gender** | Females default slightly less than males (~20% vs ~24%) |
| **Education** | University and grad school clients default less than high school clients |
| **Age** | Weak effect — slightly younger clients tend to default more |
| **Bill amounts** | Highly correlated across months (r > 0.9) — bill amount evolves slowly |
| **Payment amounts** | Much less correlated across months — payment behavior is more variable |
| **PAY history** | Cascading delays (PAY_0 → PAY_6) are strongly correlated with each other and with default |
| **No missing values** | Dataset is complete — ideal starting point for artificial missingness injection |